In [ ]:
!pip install -U kaleido

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset, random_split
import torch.nn.functional as F
import numpy as np
import gc
from tqdm import tqdm
import optuna
import joblib
import matplotlib.pyplot as plt
import os
import seaborn as sns
from IPython.display import clear_output
import optuna.visualization as vis
import json

In [ ]:
# --- CONFIGURATION ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

INPUT_DIR = '/kaggle/input/datasets/itapita/lucidrf-unet-final' 
TRAIN_X_PATH = os.path.join(INPUT_DIR, 'X_train2.npy')
TRAIN_Y_PATH = os.path.join(INPUT_DIR, 'Y_train2.npy')
VAL_X_PATH = os.path.join(INPUT_DIR, 'X_val2.npy')
VAL_Y_PATH = os.path.join(INPUT_DIR, 'Y_val2.npy')

In [ ]:
def calc_evm_percent(predictions, targets):
    """Calculates RMS EVM percentage for I/Q signals."""
    # Assuming shape (Batch, 2, L)
    error_i = predictions[:, 0, :] - targets[:, 0, :]
    error_q = predictions[:, 1, :] - targets[:, 1, :]
    
    rms_error = np.sqrt(np.mean(error_i**2 + error_q**2))
    rms_target = np.sqrt(np.mean(targets[:, 0, :]**2 + targets[:, 1, :]**2))
    
    return float((rms_error / (rms_target + 1e-10)) * 100)

In [ ]:
def calc_sinr_db(predictions, targets):
    """
    Calculates the Signal-to-Interference-Noise Ratio (SINR) in dB.
    
    Args:
        predictions (np.array): (Batch, 2, Length)
        targets (np.array): The Ground Truth Clean Signal
    Returns:
        float: The average SINR in dB across the batch.
    """
    pred_flat = predictions.reshape(predictions.shape[0], -1)
    targ_flat = targets.reshape(targets.shape[0], -1)

    # Vectorized Power Calculation (Axis 1 = across time/channels)
    signal_power = np.mean(targ_flat ** 2, axis=1)
    
    # Noise = Difference between Prediction and Target
    noise_power = np.mean((pred_flat - targ_flat) ** 2, axis=1)

    # Ratio Calculation (with epsilon for safety)
    ratio = signal_power / (noise_power + 1e-10)
    
    # Convert to dB
    sinr_db = 10 * np.log10(ratio)

    # Return scalar mean
    return float(np.mean(sinr_db))


In [ ]:
def setup_plotting_style():
    """
    Configures the global plotting style for the entire project.
    Call this once at the start of your notebook.
    """
    # Set the theme (Seaborn handles matplotlib under the hood)
    sns.set_theme(
        style="whitegrid", 
        palette="deep",
        font_scale=1.2  
    )
    
    plt.rcParams['figure.figsize'] = (10, 6)  # Default size
    plt.rcParams['savefig.dpi'] = 300         # High-res saving
    
    print("Plotting style configured")

setup_plotting_style()


In [ ]:
def plot_training_live_history(history):
    """
    Updates the Loss and SINR curves in real-time.
    """
    clear_output(wait=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot Loss
    ax1.plot(history['train_loss'], label='Train Loss', color='C0')
    ax1.plot(history['val_loss'], label='Val Loss', color='C1')
    ax1.set_title("Model Loss (MSE)")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot SINR
    ax2.plot(history['train_sinr'], label='Train SINR', color='C0')
    ax2.plot(history['val_sinr'], label='Val SINR', color='C1')
    ax2.set_title("Model Quality (SINR dB)")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("dB (Higher is Better)")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    return fig

In [ ]:
class DoubleConv(nn.Module):
    """
    (Conv1d => BN => ReLU) * 2
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class LucidRF_UNet(nn.Module):
    """
    1D U-Net for RF Signal Denoising.
    """
    def __init__(self, n_channels=2, n_classes=2, base=16):
        super(LucidRF_UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes

        # --- Encoder ---
        self.inc = DoubleConv(n_channels, base)
        self.down1 = nn.Sequential(nn.MaxPool1d(2), DoubleConv(base, base*2))
        self.down2 = nn.Sequential(nn.MaxPool1d(2), DoubleConv(base*2, base*4))
        self.down3 = nn.Sequential(nn.MaxPool1d(2), DoubleConv(base*4, base*8))
        self.down4 = nn.Sequential(nn.MaxPool1d(2), DoubleConv(base*8, base*16))

        # --- Decoder ---
        self.up1 = nn.ConvTranspose1d(base*16, base*8, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv(base*16, base*8)
        
        self.up2 = nn.ConvTranspose1d(base*8, base*4, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv(base*8, base*4)
        
        self.up3 = nn.ConvTranspose1d(base*4, base*2, kernel_size=2, stride=2)
        self.conv_up3 = DoubleConv(base*4, base*2)
        
        self.up4 = nn.ConvTranspose1d(base*2, base, kernel_size=2, stride=2)
        self.conv_up4 = DoubleConv(base*2, base)

        # --- Output ---
        self.outc = nn.Conv1d(base, n_classes, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5)
        x = torch.cat([x4, x], dim=1) # Skip connection
        x = self.conv_up1(x)

        x = self.up2(x)
        x = torch.cat([x3, x], dim=1) # Skip connection
        x = self.conv_up2(x)

        x = self.up3(x)
        x = torch.cat([x2, x], dim=1) # Skip connection
        x = self.conv_up3(x)

        x = self.up4(x)
        x = torch.cat([x1, x], dim=1) # Skip connection
        x = self.conv_up4(x)

        logits = self.outc(x)
        return logits

In [ ]:
class PhaseAwareLoss(nn.Module):
    def __init__(self, phase_weight=0.5):
        """
        Combines standard MSE with a Cosine Similarity penalty to enforce 
        phase alignment for RF constellations (like QPSK).
        """
        super().__init__()
        self.mse = nn.MSELoss()
        self.phase_weight = phase_weight

    def forward(self, y_pred, y_true):
        # Standard Amplitude/Waveform Loss (Maintains volume stability)
        mse_loss = self.mse(y_pred, y_true)
        
        # 2. Phase Alignment (Cosine Similarity)
        # y_pred and y_true are shape (Batch, 2, L) where dim=1 is the I/Q channel
        # Cosine similarity evaluates to 1 if perfectly aligned, -1 if opposite.
        cos_sim = F.cosine_similarity(y_pred, y_true, dim=1)
        
        # We want to minimize the difference from perfect alignment (1.0)
        # Taking the mean across the batch and sequence length
        phase_penalty = torch.mean(1.0 - cos_sim)
        
        # 3. Combined Objective
        total_loss = mse_loss + (self.phase_weight * phase_penalty)
        
        return total_loss

In [ ]:
class LucidRFDataset(Dataset):
    """
    PyTorch Dataset for LucidRF Denoising Task.
    
    Reads pre-split .npy tensors.
    Expected Input Shape: (N, 2, 40960) float32
    """
    def __init__(self, x_path, y_path):
        """
        Args:
            x_path (str): Path to the Noisy input .npy file
            y_path (str): Path to the Clean target .npy file
        """
        # Load data. Using mmap_mode='r' is safer for large files, 
        self.X_data = np.load(x_path, mmap_mode='r')
        self.Y_data = np.load(y_path, mmap_mode='r')
        
        # Verify alignment
        assert self.X_data.shape[0] == self.Y_data.shape[0], \
            f"Size Mismatch! X: {self.X_data.shape}, Y: {self.Y_data.shape}"


    def __len__(self):
        return self.X_data.shape[0]

    def __getitem__(self, idx):
        # Get raw samples
        # Shape expected: (2, 40960) or (40960,) complex depending on your save format
        x_sample = np.array(self.X_data[idx])
        y_sample = np.array(self.Y_data[idx])

        x_tensor = torch.from_numpy(x_sample).float()
        y_tensor = torch.from_numpy(y_sample).float()

        return x_tensor, y_tensor

In [ ]:
train_ds = LucidRFDataset(TRAIN_X_PATH, TRAIN_Y_PATH)
val_ds = LucidRFDataset(VAL_X_PATH, VAL_Y_PATH)

def objective(trial):
    # Search Space
    base_channels = trial.suggest_categorical("base_channels", [8, 16, 32])
    loss_type = trial.suggest_categorical("loss_type", ["MSE", "PhaseAware"])
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [8, 16])
    grad_clip = trial.suggest_float("grad_clip", 0.5, 2.0)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-4, log=True)


    print(f"\n{'='*40}")
    print(f"\n[Trial {trial.number}] Starting: base={base_channels}, loss={loss_type}, lr={lr:.2e}, batch_size={batch_size}", flush=True)
    print(f"{'='*40}")
    
    if loss_type == "PhaseAware":
        p_weight = trial.suggest_float("phase_weight", 0.1, 1.5)
        criterion = PhaseAwareLoss(phase_weight=p_weight).to(DEVICE)
    else:
        criterion = nn.MSELoss().to(DEVICE)

    # Setup Trial
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size)

    model = LucidRF_UNet(base=base_channels).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    best_trial_evm = float('inf')

    # 3. Mini-Training (20 epochs for HPO)
    for epoch in range(20):
        model.train()
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        # Validation
        model.eval()
        evms, sinrs = [], []
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                outputs = model(inputs)

                out_np = outputs.cpu().numpy()
                tgt_np = targets.cpu().numpy()
                evms.append(calc_evm_percent(out_np, tgt_np))
                sinrs.append(calc_sinr_db(out_np, tgt_np))
        avg_evm = np.mean(evms)
        avg_sinr = np.mean(sinrs)

        print(f"  > Epoch {epoch:02d} | EVM: {avg_evm:.2f}% | SINR: {avg_sinr:.1f} dB", flush=True)
        trial.report(avg_evm, epoch)
        if trial.should_prune():
            print(f"  !! Trial {trial.number} pruned at epoch {epoch}")
            raise optuna.exceptions.TrialPruned()
        best_trial_evm = min(best_trial_evm, avg_evm)

    return best_trial_evm

In [ ]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

# --- EXPORTING THE RESULTS (Kaggle Output) ---
# Convert to a standard CSV spreadsheet for download
results_df = study.trials_dataframe()
results_df.to_csv("/kaggle/working/lucidrf_optuna_results.csv", index=False)

report = {
    "best_evm": f"{study.best_value:.2f}%",
    "best_parameters": study.best_params,
    "total_trials": len(study.trials),
    "experiment_date": "2026-02-20"
}

with open("/kaggle/working/experiment_summary.json", "w") as f:
    json.dump(report, f, indent=4)

# Save visual charts as images
vis.plot_optimization_history(study).write_html("/kaggle/working/evm_progress.html")
vis.plot_param_importances(study).write_html("/kaggle/working/what_mattered_most.html")
vis.plot_parallel_coordinate(study).write_html("/kaggle/working/parameter_paths.html")

print(f"Optimization finished! Best EVM found: {study.best_value:.2f}%")